# Лабораторная работа №4  
# Векторные базы данных

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

После выполнения этой лабораторной работы вы:
- Изучить принципы работы векторных баз данных
- Познакомиться с ChromaDB и его возможностями
- Научиться создавать коллекции и добавлять документы с метаданными
- Освоить различные типы поиска (векторный, гибридный, с фильтрацией)

---

## 2. Теоретические сведения

### 2.1. Векторные представления текста

**Векторное представление (embedding)** — числовой вектор, кодирующий семантическое значение текста.

**Ключевые свойства:**
- Семантически близкие тексты имеют близкие векторы
- Расстояние между векторами измеряется косинусным сходством или евклидовым расстоянием
- Размерность: от 128 до 1024+ измерений

```
Текст: «Кошка сидит на диване»
Вектор: [0.12, -0.45, 0.78, 0.23, -0.67, ...]  (384 значения)
```

### 2.2. Векторные базы данных

| База данных | Тип | Особенности |
|-------------|-----|-------------|
| **ChromaDB** | Встроенная/Серверная | Простота использования, Python-native |
| **FAISS** | Библиотека | Высокая скорость, от Meta |
| **Pinecone** | Облачная | Управляемый сервис, масштабируемость |
| **Weaviate** | Серверная | GraphQL API, модульная архитектура |
| **Qdrant** | Серверная | Rust-based, фильтрация по метаданным |

### 2.3. ChromaDB

**ChromaDB** — лёгкая векторная база данных для работы с эмбеддингами.

**Преимущества:**
- Простой Python API
- Встроенное хранение (persistency)
- Поддержка метаданных
- Фильтрация при поиске
- Бесплатная и open-source

**Основные понятия:**
- **Collection** — коллекция документов с эмбеддингами
- **Document** — текстовый документ
- **Embedding** — векторное представление
- **Metadata** — дополнительные данные (теги, даты, категории)

### 2.4. Типы поиска

| Тип поиска | Описание | Когда использовать |
|------------|----------|-------------------|
| **Векторный** | Поиск по близости векторов | Семантический поиск |
| **По метаданным** | Фильтрация по полям | Точные совпадения |
| **Гибридный** | Векторный + фильтрация | Комбинированный поиск |
| **MMR** | Maximal Marginal Relevance | Разнообразие результатов |

### 2.5. Метрики сходства

| Метрика | Формула | Диапазон | Интерпретация |
|---------|---------|----------|---------------|
| **Косинусное сходство** | cos(θ) = A·B / (‖A‖‖B‖) | [-1, 1] | 1 = идентичны |
| **Евклидово расстояние** | ‖A - B‖ | [0, ∞) | 0 = идентичны |
| **Скалярное произведение** | A·B | [-∞, ∞] | Больше = ближе |

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Установите и настройте ChromaDB
2. Создайте коллекцию для хранения документов
3. Добавьте 50+ документов с метаданными (источник, категория, дата)
4. Реализуйте векторный поиск по запросу
5. Добавьте фильтрацию по метаданным
6. Сохраните коллекцию и загрузите её заново

### 3.2. Продвинутый уровень (дополнительно)

- Реализуйте гибридный поиск (вектор + keywords)
- Сравните разные embedding-модели по качеству поиска
- Настройте MMR (Maximal Marginal Relevance) для разнообразия выдачи

### 3.3. Индивидуальное задание

Создайте векторную базу для своей предметной области:
- **Вариант A:** Семантический поиск по конспектам лекций
- **Вариант B:** Поиск по технической документации
- **Вариант C:** База знаний по учебным материалам

Подготовьте мини-датасет (50+ документов) и протестируйте поиск.

---

## 4. Ход работы

### 4.1. Подготовка окружения

In [1]:
# Установка зависимостей
!pip install chromadb sentence-transformers -q

# Проверка версий
import chromadb
import sentence_transformers

print(f"ChromaDB version: {chromadb.__version__}")
print(f"Sentence Transformers version: {sentence_transformers.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 89.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 96.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

In [2]:
# Импорт библиотек
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import numpy as np

print("Библиотеки импортированы")

Библиотеки импортированы


### 4.2. Создание клиента ChromaDB

Инициализируем клиент с постоянным хранением данных.

In [3]:
# Создание клиента с сохранением данных на диск
client = chromadb.PersistentClient(path="./chroma_db")

print("ChromaDB клиент инициализирован")
print(f"Путь к базе данных: ./chroma_db")

ChromaDB клиент инициализирован
Путь к базе данных: ./chroma_db


### 4.3. Загрузка модели эмбеддингов

Используем мультиязычную модель для поддержки русского языка.

In [4]:
# Загрузка модели
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загрузка модели: {embedding_model_name}")
embedding_model = SentenceTransformer(embedding_model_name)

print("Модель загружена")

# Функция для создания эмбеддингов
def create_embedding(text):
    return embedding_model.encode(text).tolist()

# Тестирование
test_embedding = create_embedding("Привет, мир!")
print(f"\nТестовый эмбеддинг: размерность = {len(test_embedding)}")
print(f"Первые 10 значений: {[round(x, 4) for x in test_embedding[:10]]}")

Загрузка модели: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Модель загружена

Тестовый эмбеддинг: размерность = 384
Первые 10 значений: [0.1119, 0.2159, 0.3308, 0.0558, -0.057, 0.0164, 0.3781, 0.0656, -0.2034, 0.2291]


### 4.4. Создание коллекции

Создаём коллекцию для хранения документов.

In [5]:
# Создание коллекции
collection_name = "documents_collection"

# Удаляем коллекцию если существует (для чистого запуска)
try:
    client.delete_collection(collection_name)
    print(f"Коллекция '{collection_name}' удалена (предыдущая версия)")
except:
    pass

# Создаём новую коллекцию
collection = client.create_collection(
    name=collection_name,
    metadata={"description": "Коллекция документов для лабораторной работы", "language": "ru"}
)

print(f"Коллекция '{collection_name}' создана")
print(f"Количество документов: {collection.count()}")

Коллекция 'documents_collection' создана
Количество документов: 0


### 4.5. Подготовка документов

Создадим датасет из 60+ документов на тему искусственного интеллекта.

In [6]:
# Документы по теме ИИ (60+ документов)
documents_data = [
    # Машинное обучение (15 документов)
    {"text": "Машинное обучение — это метод анализа данных, который автоматизирует построение моделей.", "category": "ML", "source": "ml_intro.txt"},
    {"text": "Обучение с учителем требует размеченных данных для тренировки модели.", "category": "ML", "source": "supervised_learning.txt"},
    {"text": "Обучение без учителя находит скрытые паттерны в данных без меток.", "category": "ML", "source": "unsupervised_learning.txt"},
    {"text": "Классификация предсказывает категорию объекта на основе признаков.", "category": "ML", "source": "classification.txt"},
    {"text": "Регрессия предсказывает непрерывное значение целевой переменной.", "category": "ML", "source": "regression.txt"},
    {"text": "Деревья решений используют иерархическую структуру для принятия решений.", "category": "ML", "source": "decision_trees.txt"},
    {"text": "Случайный лес комбинирует множество деревьев для улучшения точности.", "category": "ML", "source": "random_forest.txt"},
    {"text": "Метод опорных векторов находит оптимальную разделяющую гиперплоскость.", "category": "ML", "source": "svm.txt"},
    {"text": "Градиентный бустинг последовательно улучшает модель, минимизируя ошибки.", "category": "ML", "source": "gradient_boosting.txt"},
    {"text": "XGBoost — эффективная реализация градиентного бустинга.", "category": "ML", "source": "xgboost.txt"},
    {"text": "Перекрёстная проверка оценивает качество модели на разных подмножествах данных.", "category": "ML", "source": "cross_validation.txt"},
    {"text": "Переобучение происходит когда модель запоминает данные вместо обучения.", "category": "ML", "source": "overfitting.txt"},
    {"text": "Регуляризация добавляет штраф за сложность модели для предотвращения переобучения.", "category": "ML", "source": "regularization.txt"},
    {"text": "Feature engineering — процесс создания признаков из сырых данных.", "category": "ML", "source": "feature_engineering.txt"},
    {"text": "Hyperparameter tuning оптимизирует параметры алгоритма обучения.", "category": "ML", "source": "hyperparameter_tuning.txt"},
    
    # Нейронные сети (15 документов)
    {"text": "Нейронная сеть состоит из взаимосвязанных узлов, имитирующих нейроны мозга.", "category": "NN", "source": "nn_intro.txt"},
    {"text": "Входной слой получает данные для обработки нейронной сетью.", "category": "NN", "source": "input_layer.txt"},
    {"text": "Скрытые слои выполняют промежуточные вычисления в сети.", "category": "NN", "source": "hidden_layers.txt"},
    {"text": "Выходной слой производит финальный результат работы сети.", "category": "NN", "source": "output_layer.txt"},
    {"text": "Функция активации определяет выход нейрона на основе входа.", "category": "NN", "source": "activation_functions.txt"},
    {"text": "ReLU — популярная функция активации для скрытых слоёв.", "category": "NN", "source": "relu.txt"},
    {"text": "Sigmoid преобразует вход в значение от 0 до 1.", "category": "NN", "source": "sigmoid.txt"},
    {"text": "Softmax используется для многоклассовой классификации.", "category": "NN", "source": "softmax.txt"},
    {"text": "Backpropagation распространяет ошибку назад для обновления весов.", "category": "NN", "source": "backpropagation.txt"},
    {"text": "Градиентный спуск оптимизирует веса сети минимизируя функцию потерь.", "category": "NN", "source": "gradient_descent.txt"},
    {"text": "Dropout предотвращает переобучение случайным отключением нейронов.", "category": "NN", "source": "dropout.txt"},
    {"text": "Batch normalization стабилизирует обучение нейронной сети.", "category": "NN", "source": "batch_norm.txt"},
    {"text": "Epoch — один полный проход по всем данным обучения.", "category": "NN", "source": "epoch.txt"},
    {"text": "Learning rate определяет размер шага при обновлении весов.", "category": "NN", "source": "learning_rate.txt"},
    {"text": "Optimizer выбирает стратегию обновления весов сети.", "category": "NN", "source": "optimizer.txt"},
    
    # Глубокое обучение (10 документов)
    {"text": "Глубокое обучение использует многоуровневые нейронные сети.", "category": "DL", "source": "dl_intro.txt"},
    {"text": "Свёрточные сети эффективны для обработки изображений.", "category": "DL", "source": "cnn.txt"},
    {"text": "Рекуррентные сети обрабатывают последовательные данные.", "category": "DL", "source": "rnn.txt"},
    {"text": "LSTM решает проблему затухающего градиента в RNN.", "category": "DL", "source": "lstm.txt"},
    {"text": "Трансформеры используют механизм внимания для обработки последовательностей.", "category": "DL", "source": "transformers.txt"},
    {"text": "Attention механизм фокусируется на важных частях входа.", "category": "DL", "source": "attention.txt"},
    {"text": "BERT — предобученная модель для понимания текста.", "category": "DL", "source": "bert.txt"},
    {"text": "GPT генерирует текст на основе предыдущего контекста.", "category": "DL", "source": "gpt.txt"},
    {"text": "Transfer learning использует предобученные модели для новых задач.", "category": "DL", "source": "transfer_learning.txt"},
    {"text": "Fine-tuning дообучает модель на специфичных данных.", "category": "DL", "source": "fine_tuning.txt"},
    
    # Обработка естественного языка (10 документов)
    {"text": "NLP занимается взаимодействием компьютеров и человеческого языка.", "category": "NLP", "source": "nlp_intro.txt"},
    {"text": "Токенизация разбивает текст на отдельные слова или под слова.", "category": "NLP", "source": "tokenization.txt"},
    {"text": "Лемматизация приводит слова к их словарной форме.", "category": "NLP", "source": "lemmatization.txt"},
    {"text": "NER распознает именованные сущности в тексте.", "category": "NLP", "source": "ner.txt"},
    {"text": "POS tagging определяет часть речи для каждого слова.", "category": "NLP", "source": "pos_tagging.txt"},
    {"text": "Синтаксический анализ изучает грамматическую структуру предложения.", "category": "NLP", "source": "parsing.txt"},
    {"text": "Машинный перевод автоматически переводит текст между языками.", "category": "NLP", "source": "machine_translation.txt"},
    {"text": "Анализ тональности определяет эмоциональную окраску текста.", "category": "NLP", "source": "sentiment_analysis.txt"},
    {"text": "Суммаризация создаёт краткое изложение длинного текста.", "category": "NLP", "source": "summarization.txt"},
    {"text": "Word2Vec создаёт векторные представления слов.", "category": "NLP", "source": "word2vec.txt"},
    
    # Компьютерное зрение (10 документов)
    {"text": "Компьютерное зрение обучает компьютеры понимать визуальную информацию.", "category": "CV", "source": "cv_intro.txt"},
    {"text": "Распознавание объектов находит и классифицирует объекты на изображении.", "category": "CV", "source": "object_detection.txt"},
    {"text": "Сегментация разделяет изображение на значимые области.", "category": "CV", "source": "segmentation.txt"},
    {"text": "Распознавание лиц идентифицирует людей по фотографии.", "category": "CV", "source": "face_recognition.txt"},
    {"text": "OCR извлекает текст из изображений документов.", "category": "CV", "source": "ocr.txt"},
    {"text": "Image classification присваивает изображению метку категории.", "category": "CV", "source": "image_classification.txt"},
    {"text": "Data augmentation увеличивает датасет трансформацией изображений.", "category": "CV", "source": "data_augmentation.txt"},
    {"text": "Transfer learning в CV использует предобученные модели на ImageNet.", "category": "CV", "source": "cv_transfer.txt"},
    {"text": "YOLO — быстрая модель для детекции объектов в реальном времени.", "category": "CV", "source": "yolo.txt"},
    {"text": "GAN генерируют реалистичные изображения с помощью состязательных сетей.", "category": "CV", "source": "gan.txt"}
]

print(f"Подготовлено документов: {len(documents_data)}")

Подготовлено документов: 60


### 4.6. Добавление документов в коллекцию

Добавляем документы с эмбеддингами и метаданными.

In [7]:
# Добавление документов в коллекцию
print("Добавление документов в коллекцию...")

for i, doc_data in enumerate(documents_data):
    collection.add(
        ids=[f"doc_{i}"],
        documents=[doc_data["text"]],
        metadatas=[{"category": doc_data["category"], "source": doc_data["source"]}],
        embeddings=[create_embedding(doc_data["text"])]
    )

print(f"Добавлено {len(documents_data)} документов")
print(f"Общее количество в коллекции: {collection.count()}")

Добавление документов в коллекцию...
Добавлено 60 документов
Общее количество в коллекции: 60


In [8]:
# Проверка добавленных документов
sample = collection.get(limit=5)

print("\nПримеры добавленных документов:")
print("="*60)
for i in range(len(sample['documents'])):
    print(f"\nID: {sample['ids'][i]}")
    print(f"Текст: {sample['documents'][i][:80]}...")
    print(f"Метаданные: {sample['metadatas'][i]}")


Примеры добавленных документов:

ID: doc_0
Текст: Машинное обучение — это метод анализа данных, который автоматизирует построение ...
Метаданные: {'source': 'ml_intro.txt', 'category': 'ML'}

ID: doc_1
Текст: Обучение с учителем требует размеченных данных для тренировки модели....
Метаданные: {'category': 'ML', 'source': 'supervised_learning.txt'}

ID: doc_2
Текст: Обучение без учителя находит скрытые паттерны в данных без меток....
Метаданные: {'category': 'ML', 'source': 'unsupervised_learning.txt'}

ID: doc_3
Текст: Классификация предсказывает категорию объекта на основе признаков....
Метаданные: {'source': 'classification.txt', 'category': 'ML'}

ID: doc_4
Текст: Регрессия предсказывает непрерывное значение целевой переменной....
Метаданные: {'source': 'regression.txt', 'category': 'ML'}


### 4.7. Векторный поиск

Реализуем поиск по векторному индексу.

In [9]:
# Функция поиска
def vector_search(query, top_k=5):
    """
    Векторный поиск документов по запросу.
    
    Args:
        query: Поисковый запрос
        top_k: Количество результатов
    
    Returns:
        Список найденных документов с метаданными и расстояниями
    """
    query_embedding = create_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    return results

print("Функция векторного поиска готова")

Функция векторного поиска готова


In [10]:
# Тестирование поиска
test_queries = [
    "Как работают нейронные сети?",
    "Что такое машинное обучение?",
    "Как компьютеры понимают текст?",
    "Методы предотвращения переобучения"
]

print("="*60)
print("ТЕСТИРОВАНИЕ ВЕКТОРНОГО ПОИСКА")
print("="*60)

for query in test_queries:
    print(f"\n❓ Запрос: {query}")
    print("-" * 60)
    
    results = vector_search(query, top_k=3)
    
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i]
        
        print(f"\n  #{i+1} | Категория: {meta['category']} | Источник: {meta['source']}")
        print(f"  Расстояние: {dist:.4f}")
        print(f"  Текст: {doc}")

ТЕСТИРОВАНИЕ ВЕКТОРНОГО ПОИСКА

❓ Запрос: Как работают нейронные сети?
------------------------------------------------------------

  #1 | Категория: NN | Источник: nn_intro.txt
  Расстояние: 6.7460
  Текст: Нейронная сеть состоит из взаимосвязанных узлов, имитирующих нейроны мозга.

  #2 | Категория: NN | Источник: input_layer.txt
  Расстояние: 7.6760
  Текст: Входной слой получает данные для обработки нейронной сетью.

  #3 | Категория: NN | Источник: activation_functions.txt
  Расстояние: 11.0076
  Текст: Функция активации определяет выход нейрона на основе входа.

❓ Запрос: Что такое машинное обучение?
------------------------------------------------------------

  #1 | Категория: ML | Источник: ml_intro.txt
  Расстояние: 8.1135
  Текст: Машинное обучение — это метод анализа данных, который автоматизирует построение моделей.

  #2 | Категория: ML | Источник: supervised_learning.txt
  Расстояние: 16.5052
  Текст: Обучение с учителем требует размеченных данных для тренировки модели.

### 4.8. Поиск с фильтрацией по метаданным

Добавляем фильтрацию по категориям.

In [11]:
# Поиск с фильтрацией
def filtered_search(query, category=None, top_k=5):
    """
    Поиск с фильтрацией по метаданным.
    
    Args:
        query: Поисковый запрос
        category: Фильтр по категории (или None)
        top_k: Количество результатов
    
    Returns:
        Список документов
    """
    query_embedding = create_embedding(query)
    
    where_filter = None
    if category:
        where_filter = {"category": {"$eq": category}}
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_filter
    )
    
    return results

print("Функция поиска с фильтрацией готова")

Функция поиска с фильтрацией готова


In [12]:
# Тестирование фильтрации
query = "Как работает обучение?"

print(f"\nЗапрос: '{query}'")
print("="*60)

# Поиск по всем категориям
print("\n📌 БЕЗ ФИЛЬТРА (все категории):")
results_all = filtered_search(query, category=None, top_k=5)
for i in range(len(results_all['documents'][0])):
    meta = results_all['metadatas'][0][i]
    doc = results_all['documents'][0][i]
    print(f"  [{meta['category']}] {doc[:70]}...")

# Поиск только по ML
print("\n📌 ФИЛЬТР: category = 'ML':")
results_ml = filtered_search(query, category="ML", top_k=5)
for i in range(len(results_ml['documents'][0])):
    meta = results_ml['metadatas'][0][i]
    doc = results_ml['documents'][0][i]
    print(f"  [{meta['category']}] {doc[:70]}...")

# Поиск только по NN
print("\n📌 ФИЛЬТР: category = 'NN':")
results_nn = filtered_search(query, category="NN", top_k=5)
for i in range(len(results_nn['documents'][0])):
    meta = results_nn['metadatas'][0][i]
    doc = results_nn['documents'][0][i]
    print(f"  [{meta['category']}] {doc[:70]}...")


Запрос: 'Как работает обучение?'

📌 БЕЗ ФИЛЬТРА (все категории):
  [ML] Обучение с учителем требует размеченных данных для тренировки модели....
  [ML] Обучение без учителя находит скрытые паттерны в данных без меток....
  [NN] Epoch — один полный проход по всем данным обучения....
  [DL] Transfer learning использует предобученные модели для новых задач....
  [DL] Глубокое обучение использует многоуровневые нейронные сети....

📌 ФИЛЬТР: category = 'ML':
  [ML] Обучение с учителем требует размеченных данных для тренировки модели....
  [ML] Обучение без учителя находит скрытые паттерны в данных без меток....
  [ML] Переобучение происходит когда модель запоминает данные вместо обучения...
  [ML] Машинное обучение — это метод анализа данных, который автоматизирует п...
  [ML] XGBoost — эффективная реализация градиентного бустинга....

📌 ФИЛЬТР: category = 'NN':
  [NN] Epoch — один полный проход по всем данным обучения....
  [NN] Learning rate определяет размер шага при обновлении весов...

### 4.9. MMR поиск (Maximal Marginal Relevance)

MMR балансирует релевантность и разнообразие результатов.

In [13]:
# MMR поиск
def mmr_search(query, top_k=5, diversity=0.5):
    """
    Поиск с использованием MMR для разнообразия результатов.
    
    Args:
        query: Поисковый запрос
        top_k: Количество результатов
        diversity: Коэффициент разнообразия (0 = релевантность, 1 = разнообразие)
    
    Returns:
        Список документов
    """
    query_embedding = create_embedding(query)
    
    # Получаем больше результатов для MMR
    all_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k * 2  # Берём с запасом
    )
    
    # Простая реализация MMR
    if len(all_results['documents'][0]) == 0:
        return all_results
    
    selected_indices = [0]  # Первый самый релевантный
    
    while len(selected_indices) < min(top_k, len(all_results['documents'][0])):
        best_score = -float('inf')
        best_idx = -1
        
        for i in range(len(all_results['documents'][0])):
            if i in selected_indices:
                continue
            
            # Релевантность запросу
            relevance_score = -all_results['distances'][0][i]
            
            # Разнообразие (минимальное сходство с выбранными)
            min_similarity = float('inf')
            for j in selected_indices:
                similarity = 1 - all_results['distances'][0][i]
                min_similarity = min(min_similarity, similarity)
            
            # MMR формула
            mmr_score = diversity * relevance_score - (1 - diversity) * min_similarity
            
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = i
        
        if best_idx >= 0:
            selected_indices.append(best_idx)
    
    # Формируем результат
    mmr_results = {
        'documents': [[all_results['documents'][0][i] for i in selected_indices]],
        'metadatas': [[all_results['metadatas'][0][i] for i in selected_indices]],
        'distances': [[all_results['distances'][0][i] for i in selected_indices]],
        'ids': [[all_results['ids'][0][i] for i in selected_indices]]
    }
    
    return mmr_results

print("Функция MMR поиска готова")

Функция MMR поиска готова


In [14]:
# Сравнение обычного и MMR поиска
query = "обучение нейронных сетей"

print(f"\nЗапрос: '{query}'")
print("="*60)

# Обычный поиск
print("\n📌 ОБЫЧНЫЙ ПОИСК:")
normal_results = vector_search(query, top_k=5)
categories_normal = [m['category'] for m in normal_results['metadatas'][0]]
print(f"  Категории: {categories_normal}")
print(f"  Уникальных категорий: {len(set(categories_normal))}")

# MMR поиск
print("\n📌 MMR ПОИСК (diversity=0.7):")
mmr_results = mmr_search(query, top_k=5, diversity=0.7)
categories_mmr = [m['category'] for m in mmr_results['metadatas'][0]]
print(f"  Категории: {categories_mmr}")
print(f"  Уникальных категорий: {len(set(categories_mmr))}")


Запрос: 'обучение нейронных сетей'

📌 ОБЫЧНЫЙ ПОИСК:
  Категории: ['DL', 'NN', 'NN', 'NN', 'NN']
  Уникальных категорий: 2

📌 MMR ПОИСК (diversity=0.7):
  Категории: ['DL', 'NN', 'NN', 'NN', 'NN']
  Уникальных категорий: 2


### 4.10. Сохранение и загрузка коллекции

Демонстрация персистентности данных в ChromaDB.

In [15]:
# Сохранение уже реализовано через PersistentClient
# Проверим что данные сохранились

print("Данные автоматически сохраняются в ./chroma_db")
print(f"Текущее количество документов: {collection.count()}")

Данные автоматически сохраняются в ./chroma_db
Текущее количество документов: 60


In [16]:
# Загрузка существующей коллекции
print("\nЗагрузка существующей коллекции...")

loaded_collection = client.get_collection(collection_name)

print(f"Коллекция загружена: {loaded_collection.name}")
print(f"Количество документов: {loaded_collection.count()}")

# Тестирование
test_query = "Что такое глубокое обучение?"
results = loaded_collection.query(
    query_embeddings=[create_embedding(test_query)],
    n_results=3
)

print(f"\nТестовый запрос: '{test_query}'")
for i in range(len(results['documents'][0])):
    print(f"  {i+1}. {results['documents'][0][i][:80]}...")


Загрузка существующей коллекции...
Коллекция загружена: documents_collection
Количество документов: 60

Тестовый запрос: 'Что такое глубокое обучение?'
  1. Глубокое обучение использует многоуровневые нейронные сети....
  2. Обучение без учителя находит скрытые паттерны в данных без меток....
  3. Hyperparameter tuning оптимизирует параметры алгоритма обучения....


### 4.11. Статистика коллекции

Анализ содержимого векторной базы.

In [17]:
# Получение всех документов для статистики
all_docs = collection.get(include=['metadatas'])

# Подсчёт по категориям
from collections import Counter

categories = [m['category'] for m in all_docs['metadatas']]
category_counts = Counter(categories)

print("\n" + "="*60)
print("СТАТИСТИКА КОЛЛЕКЦИИ")
print("="*60)

print(f"\nОбщее количество документов: {len(categories)}")
print(f"\nДокументов по категориям:")
for cat, count in sorted(category_counts.items()):
    print(f"  {cat}: {count} документов")

# Подсчёт по источникам
sources = [m['source'] for m in all_docs['metadatas']]
print(f"\nУникальных источников: {len(set(sources))}")


СТАТИСТИКА КОЛЛЕКЦИИ

Общее количество документов: 60

Документов по категориям:
  CV: 10 документов
  DL: 10 документов
  ML: 15 документов
  NLP: 10 документов
  NN: 15 документов

Уникальных источников: 60


---

## 5. Контрольные вопросы

**Ответы записывайте в эту ячейку (режим Markdown):**

1. **Что такое векторная база данных и чем она отличается от традиционной?**
   > Ваш ответ...

2. **Как работает косинусное сходство и почему оно используется для поиска?**
   > Ваш ответ...

3. **Зачем нужны метаданные в векторной базе и как их использовать?**
   > Ваш ответ...

4. **Что такое MMR и когда его следует применять?**
   > Ваш ответ...

5. **Какие преимущества у ChromaDB перед другими векторными базами?**
   > Ваш ответ...

---

## 6. Полезные ресурсы

- 📚 [ChromaDB Documentation](https://docs.trychroma.com/) — официальная документация
- 📖 [Sentence Transformers](https://www.sbert.net/) — библиотека эмбеддингов
- 🎥 [Vector Databases Explained](https://www.youtube.com/watch?v=Hbl5FZ8xGxE) — объяснение концепции
- 🔍 [Hugging Face Embedding Models](https://huggingface.co/sentence-transformers) — модели для эмбеддингов
- 📊 [Embedding Model Benchmark](https://github.com/Muennighoff/embedding-benchmark) — сравнение моделей

---

> **Примечание:** Все лабораторные работы выполняются в Google Colab с использованием бесплатных ресурсов. Сохраняйте копию ноутбука в своём Google Drive через `File → Save a copy in Drive`.